# Data Loading

## Loading Cyber APA Data into Neo4j

It is assumed that you have loaded the [Cyber VPEM use case data](https://github.com/pedroleitao-neo4j/cyber-vpem) into Neo4j. This notebook loads additional data required for the **Cyber Attack Path Analysis (APA)** use case, specifically focusing on **lateral movement** relationships between internal assets.

### Lateral Movement Scenario

Lateral movement refers to the techniques cybercriminals use after gaining an initial foothold to navigate deeper into a network, escalate privileges, and reach high-value targets.

In this scenario, we simulate a multi-stage breach:

1. **Initial Entry:** An attacker compromises an internet-facing **ComputeInstance** (e.g., `api-gateway-01`) via a known vulnerability.
2. **Internal Reconnaissance:** Once inside, the attacker maps the internal network to identify adjacent systems and organizational hierarchies.
3. **Sideways Progression:** The attacker moves "sideways" through the network, hopping from the compromised gateway to internal servers that are not directly exposed to the public internet.
4. **Final Objective:** The attacker ultimately targets a "Crown Jewel" asset, such as a **CoreBankingDB**, to perform data exfiltration or sabotage.

### Synthetic Data Extension

To model this scenario, we extend the graph with a new relationship type: `CAN_REACH`. While standard vulnerability scanners identify bugs, they often miss the logical network paths that connect devices.

The data loading method `create_lateral_movement_data()` performs the following actions:

* **Infrastructure Nodes:** Ensures the existence of `ComputeInstance` nodes for the gateway (`i-0001`), an internal worker (`i-0002`), and the protected database (`i-9999`).
* **Logical Connectivity:** Establishes `CAN_REACH` edges between these instances, representing open network routes (e.g., port 8080 or 5432) that an attacker can exploit.
* **Impact Mapping:** Connects the final infrastructure node to a high-tier `Application` (P0) to calculate the potential business impact.

By explicitly modeling these internal connections as a **Security Knowledge Graph**, defenders can visualize and preempt the specific routes an attacker would follow to reach critical data.

In [8]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")

def create_lateral_movement_data():
    query = """
    // Create a "Crown Jewel" Internal Database
    MERGE (db:Application {name: 'CoreBankingDB', tier: 'P0'})
    MERGE (db_ins:ComputeInstance {id: 'i-9999', name: 'prod-db-internal', private_ip: '10.0.5.1'})
    MERGE (db)-[:HOSTED_ON]->(db_ins)
    
    // Transition from MERGE to MATCH requires WITH
    WITH db_ins
    
    // Create an Internal Network Path
    // Scenario A server (gateway) can reach the Scenario B server (worker)
    MATCH (insA:ComputeInstance {id: 'i-0001'})
    MATCH (insB:ComputeInstance {id: 'i-0002'})
    MERGE (insA)-[:CAN_REACH {port: 8080}]->(insB)
    
    WITH insB, db_ins
    
    // Scenario B server (worker) can reach the Crown Jewel DB
    MERGE (insB)-[:CAN_REACH {port: 5432}]->(db_ins)
    """
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB) as session:
            session.run(query)
    print("Lateral movement paths successfully added to the VPEM graph.")

In [9]:
create_lateral_movement_data()

Lateral movement paths successfully added to the VPEM graph.


With the data loaded, we can now proceed to analyze potential attack paths in the [next notebook](apa.ipynb).